In [ ]:
#| default_exp fixtures

# Disposable notebook fixtures

> Own scratch paths and test notebooks for one example or test lifetime.

In [ ]:
#| export
import json, shutil, tempfile
from contextlib import contextmanager, nullcontext
from pathlib import Path
from fastcore.basics import in_jupyter
from fastcore.nbio import mk_cell, new_nb, write_nb

In [ ]:
from fastcore.test import test_eq
from fastcore.nbio import read_nb

#| export
## Demo scratch files

#| export
Examples and tests in later notebooks need small notebooks they can safely mutate. Scratch helpers create disposable artifacts in the system temporary directory and remove them after use, so notebook test runners never create repository data folders. `tool_notebook` builds the shared reader, executor, reviewer, and example shape in code.

`remove_demo_path` cleans up an owned scratch file and tolerates repeated cleanup.

In [ ]:
#| export
def remove_demo_path(path):
    'Remove a disposable file or directory if present and return its path.'
    path = Path(path)
    if path.is_dir(): shutil.rmtree(path)
    elif path.exists(): path.unlink()
    return path

`demo_path` names a scratch artifact in the system temporary directory by default and optionally removes any previous copy. It only creates the parent folder; helpers that write content decide what belongs at the path. An explicit `base` remains available when an example needs a chosen location.

Use `demo_path_context` when a test needs an arbitrary scratch file or directory and should clean it up after the block. That keeps cleanup policy in one helper instead of repeating `try`/`finally` in every test.

In [ ]:
#| export
def demo_path(name, base=None, reset=True):
    'Return a scratch path in the system temporary directory or `base`, optionally resetting it.'
    base = Path(tempfile.gettempdir()) / "nbskill" if base is None else Path(base)
    path = base / name
    if reset: remove_demo_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
obsolete = demo_path("obsolete.txt")
obsolete.write_text("scratch")
remove_demo_path(obsolete)
assert not obsolete.exists()

#| export
`demo_path` reserves a scratch filename and creates its parent directory. It does not create the file. The path stays available to whichever helper writes the example content:

In [ ]:
#| exportd
scratch_path = demo_path("foundation_scratch_example.txt")
assert scratch_path.parent.exists() and not scratch_path.exists()
scratch_path.name

Use `demo_path_context` when a lesson owns a disposable file or directory. Cleanup runs on both success and failure; the yielded path is not retained after the block.

In [ ]:
#| export
@contextmanager
def demo_path_context(name, base=None, reset=True):
    'Yield a demo path and remove it when the block exits.'
    path = demo_path(name, base=base, reset=reset)
    try: yield path
    finally: remove_demo_path(path)

In [ ]:
#| exportd
with demo_path_context("foundation_demo_path_context") as context_path:
    context_path.write_text("temporary evidence", encoding="utf-8")
    assert context_path.read_text(encoding="utf-8") == "temporary evidence"
assert not context_path.exists()
dict(name=context_path.name, exists_after_context=context_path.exists())

Notebook examples can run from the repository root or from `nbs/`. `path_candidates` preserves that lookup order and leaves absolute paths unchanged.

In [ ]:
#| export
def path_candidates(path):
    "Return cwd-relative path candidates for root and notebook-dir execution."
    pth = Path(str(path)).expanduser()
    if pth.is_absolute(): return [pth]
    parts = pth.parts
    in_notebook_dir = parts and parts[0] == Path.cwd().name and (in_jupyter() or Path.cwd().name == "nbs")
    if not in_notebook_dir: return [pth]
    stripped = Path(*parts[1:]) if len(parts) > 1 else Path(".")
    return [pth, stripped]

In [ ]:
_path_candidate_rows = path_candidates("nbs/00_foundation.ipynb")
expected_candidates = [Path("nbs/00_foundation.ipynb")]
if Path.cwd().name == "nbs": expected_candidates.append(Path("00_foundation.ipynb"))
assert _path_candidate_rows == expected_candidates
assert path_candidates(Path("/tmp/example.ipynb")) == [Path("/tmp/example.ipynb")]

Notebook helpers should fail gently when a path is missing, unreadable, or shaped like ordinary JSON instead of a Jupyter notebook. `is_valid_ipynb` keeps that contract small and dependency-free.

In [ ]:
#| exporti
#| hide
_VALID_CELL_TYPES = {"code", "markdown", "raw"}


def _is_valid_notebook_cell(cell):
    if not isinstance(cell, dict): return False
    if cell.get("cell_type") not in _VALID_CELL_TYPES: return False
    if not isinstance(cell.get("metadata"), dict): return False
    source = cell.get("source")
    if not isinstance(source, (str, list)): return False
    if isinstance(source, list) and not all(isinstance(line, str) for line in source): return False
    if cell["cell_type"] == "code":
        if "outputs" not in cell or not isinstance(cell.get("outputs"), list): return False
        execution_count = cell.get("execution_count")
        if execution_count is not None and not isinstance(execution_count, int): return False
    return True

In [ ]:
#| export
def is_valid_ipynb(path):
    'Return whether a file has the required notebook and cell structure.'
    try: data = json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError): return False
    if not isinstance(data, dict): return False
    if not isinstance(data.get("nbformat"), int): return False
    if not isinstance(data.get("nbformat_minor"), int): return False
    if not isinstance(data.get("metadata"), dict): return False
    cells = data.get("cells")
    return isinstance(cells, list) and all(_is_valid_notebook_cell(cell) for cell in cells)

`write_demo_notebook` owns the whole lifetime of a scratch notebook: create it, yield its path to the example, then remove it even if the example fails.

`write_demo_notebook` owns a unique temporary directory when `base` is omitted. Nested lessons can use the same filename without overwriting each other. Isolation also prevents recursive inspection from discovering another concurrently running lesson's files during cleanup. An explicit `base` retains the caller-selected directory.

In [ ]:
#| export
@contextmanager
def write_demo_notebook(name, cells=None, base=None, reset=True):
    "Write a disposable notebook with an isolated default directory and clean it up."
    owner = tempfile.TemporaryDirectory(prefix="nbskill-demo-") if base is None else nullcontext(base)
    with owner as root, demo_path_context(name, base=root, reset=reset) as path:
        cells = cells or [
            mk_cell("## Demo notebook\nThis tiny notebook gives nbskill tools something real to inspect.", cell_type="markdown"),
            mk_cell("#| export\ndef demo_answer():\n    return 42"),
            mk_cell("assert demo_answer() == 42")]
        write_nb(new_nb(cells), path)
        yield path

In [ ]:
#| hide
with write_demo_notebook("isolated_lesson.ipynb") as outer:
    original = outer.read_bytes()
    with write_demo_notebook("isolated_lesson.ipynb") as inner: assert outer.parent != inner.parent and outer.exists() and inner.exists()
    assert outer.exists() and outer.read_bytes() == original
assert not outer.parent.exists()

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as chosen_base:
    with write_demo_notebook("explicit.ipynb", base=chosen_base) as explicit: assert explicit.parent == Path(chosen_base)
    assert Path(chosen_base).exists() and not explicit.exists()

`tool_notebook` keeps the shared reader, executor, reviewer, and test shape in code. Each test writes a fresh notebook and removes it afterwards.

In [ ]:
#| export
def tool_notebook():
    "Return the canonical notebook used by tool tests and examples."
    cells = [
        mk_cell("# nbskill tool notebook\n\nA tiny notebook for disposable tool tests.", cell_type="markdown", id="tool-title"),
        mk_cell("answer = 42", id="tool-answer"),
        mk_cell("from fastcore.test import test_eq", id="tool-imports"),
        mk_cell("## Arithmetic\n\n`double_answer` gives readers and symbol tools a public function to find.", cell_type="markdown", id="tool-arithmetic-doc"),
        mk_cell("#| export\ndef double_answer(value):\n    \"Return twice the input value.\"\n    return value * 2", id="tool-double-answer"),
        mk_cell("double_answer(answer)", id="tool-example"),
        mk_cell("test_eq(double_answer(21), 42)", id="tool-test")]
    nb = new_nb(cells)
    nb.metadata["nbskill"] = {"demo": "tool", "version": 1}
    return nb

`tool_notebook` provides a small notebook with stable addresses for examples and tool tests.

In [ ]:
tool_example = tool_notebook()
assert tool_example.cells[0].id == "tool-title"
[cell.id for cell in tool_example.cells]

`write_tool_notebook` gives each example and test an isolated copy of the canonical tool notebook.

In [ ]:
#| export
@contextmanager
def write_tool_notebook(name="tool.ipynb", base=None):
    "Write the canonical tool notebook for one example or test."
    path = demo_path(name, base=base)
    write_nb(tool_notebook(), path)
    try: yield path
    finally: remove_demo_path(path)

In [ ]:
with write_tool_notebook("fixture_tour.ipynb") as fixture_path:
    fixture_cells = read_nb(fixture_path).cells
    assert fixture_cells[0].id == "tool-title"
[cell.id for cell in fixture_cells]

`write_demo_file` keeps temporary conversion inputs out of the repository.

In [ ]:
#| export
@contextmanager
def write_demo_file(name, source, base=None):
    "Write one disposable text file for an example or test."
    path = demo_path(name, base=base)
    path.write_text(source, encoding="utf-8")
    try: yield path
    finally: remove_demo_path(path)

In [ ]:
with write_demo_file("lesson.py", "answer = 42") as lesson_path:
    lesson_source = lesson_path.read_text()
    assert lesson_source == "answer = 42"
lesson_source

In [ ]:
#| hide
tool_ids = [cell.id for cell in tool_notebook().cells]
test_eq(tool_ids[:3], ["tool-title", "tool-answer", "tool-imports"])
test_eq(tool_ids[-1], "tool-test")

In [ ]:
#| hide
with write_tool_notebook("tool_test.ipynb") as path:
    test_eq(path.exists(), True)
    test_eq("answer = 42" in path.read_text(encoding="utf-8"), True)
test_eq(path.exists(), False)

In [ ]:
#| hide
with write_demo_file("tool.py", "answer = 42") as path: test_eq(path.read_text(encoding="utf-8"), "answer = 42")
test_eq(path.exists(), False)

#| export
`write_demo_notebook` creates a real temporary `.ipynb` and removes it when the context closes. Examples can check the actual file format and cleanup without leaving repository artifacts:

In [ ]:
#| exportd
with write_demo_notebook("00_foundation_tour.ipynb") as path:
    demo_nb = read_nb(path)
    assert is_valid_ipynb(path)
    fixture_summary = dict(cells=len(demo_nb.cells), types=[cell.cell_type for cell in demo_nb.cells])
assert not path.exists()
fixture_summary

In [ ]:
with write_demo_notebook("00_foundation_bad.ipynb") as bad_path:
    bad_path.write_text("{}", encoding="utf-8")
    assert not is_valid_ipynb(bad_path)
    assert not is_valid_ipynb(bad_path.with_suffix(".missing.ipynb"))